# EMPro Driven Modal Simulation of an Interdigital Capacitor

This notebook demonstrates how to set up and run a driven modal (S-parameter)
simulation of an interdigital capacitor using Keysight EMPro Python interface.

**Prerequisites:**
- Keysight EMPro 2026 installed (requires license)
- Run this notebook within the EMPro Python environment

**References:**
- Keysight EMPro Documentation (online)
- Interdigital Capacitor Theory: {cite:p}`leizhuAccurateCircuitModel2000`

## Setup and Imports

In [ ]:
import sys

if "google.colab" in sys.modules:
    import subprocess

    print("Running in Google Colab. Installing quantum-rf-pdk...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "qpdk[models] @ git+https://github.com/gdsfactory/quantum-rf-pdk.git",
    ])

In [ ]:
import empro
import gdsfactory as gf

from qpdk import PDK
from qpdk.cells.capacitor import interdigital_capacitor
from qpdk.cells.waveguides import straight_open
from qpdk.simulation import EMPro, prepare_component_for_aedt
from qpdk.tech import coplanar_waveguide

PDK.activate()

# Create an interdigital capacitor
cpw_width, cpw_gap = 10, 6
cross_section = coplanar_waveguide(width=cpw_width, gap=cpw_gap)
idc_component = interdigital_capacitor(
    fingers=6,
    finger_length=20.0,
    finger_gap=2.0,
    thickness=5.0,
    cross_section=cross_section,
)

# Attach straight open waveguides for port feeding
c = gf.Component(name="idc_with_feeds")
idc_ref = c << idc_component
open_wvg = straight_open(length=5, cross_section=cross_section)

feed1 = c << open_wvg
feed1.connect("o1", idc_ref.ports["o1"])
c.add_port("o1", port=feed1.ports["o2"])

feed2 = c << open_wvg
feed2.connect("o1", idc_ref.ports["o2"])
c.add_port("o2", port=feed2.ports["o2"])

idc_component = c

## Initialize EMPro Project

Connect to the active EMPro project and clear existing content.

In [ ]:
# Clear active project
empro.activeProject.clear()

# Initialize EMPro wrapper
emp_sim = EMPro(empro.activeProject)

print("EMPro active project initialized")

## Build EMPro Model

Import the gdsfactory component geometry into EMPro.

In [ ]:
# Prepare component for export
prepared_component = prepare_component_for_aedt(idc_component, margin_draw=50)

# Add materials to EMPro
emp_sim.add_materials()

# Import the component geometry via extrusion
created_parts = emp_sim.import_component(prepared_component)
print(f"Imported {len(created_parts)} parts into EMPro")

# Add substrate below the component
substrate_name = emp_sim.add_substrate(
    prepared_component,
    thickness=500.0,
    material="silicon",
)
print(f"Created substrate: {substrate_name}")

# Add air region
air_region_name = emp_sim.add_air_region(
    prepared_component,
    height=500.0,
    substrate_thickness=500.0,
    material="Air",
)
print(f"Created air region: {air_region_name}")

## Create Lumped Ports

Add lumped ports at both ends of the capacitor.

In [ ]:
print("Creating lumped ports.")
emp_sim.add_lumped_ports(prepared_component.ports)
for port in prepared_component.ports:
    print(f"  Created port: {port.name}")

## Configure FEM Simulation

Set up the FEM solver with a frequency sweep.

In [ ]:
# Configure FEM simulation (1 GHz to 5 GHz)
emp_sim.setup_fem_simulation(start_freq=1e9, stop_freq=5e9, num_points=101)

print("FEM simulation configured (1-5 GHz)")

## Run Simulation

Execute the simulation.

In [ ]:
print("Starting EMPro simulation...")
# Note: This will block until simulation is complete if run in EMPro
# For demonstration, we'll just trigger it
# sim = emp_sim.run_simulation(wait=True)
# print(f"Simulation status: {sim.status}")

## Summary

This notebook demonstrated the workflow for Keysight EMPro integration:

1. **Component Creation**: Defined an interdigital capacitor with gdsfactory.
2. **EMPro Connection**: Connected to the active project via the `EMPro` wrapper.
3. **Geometry Import**: Extruded 2D layouts into 3D parts and assigned materials.
4. **Simulation Setup**: Added substrate, air region, and lumped ports.
5. **Solver Configuration**: Configured the FEM engine and frequency sweep.